# 23 - nnU-Net with 10% labels, run 1

Supervised nnU-Net on 23 frames, then self-training with the pseudo-labels of that same model on the r10 pool. Replicates cells A, F and G of `02_nnunet_baseline.ipynb`. Run all; every stage is resumable.


In [ ]:
# === Cell 1: run id, Drive, nnU-Net, environment ===
RUN_ID = 1   # the only line that differs between the three notebooks

from google.colab import drive
drive.mount('/content/drive')

!pip install -q nnunetv2

import os, sys, json, importlib.metadata as md
import torch

BASE = '/content/drive/MyDrive/UNM_vertebras_seg_v3'
WORKSPACE = os.path.join(BASE, 'nnunet_workspace')
RESULTS_BASE = os.path.join(BASE, 'nnunet_baseline_results')
TAG = f'f10_run{RUN_ID}'

os.environ['nnUNet_raw'] = os.path.join(WORKSPACE, 'nnUNet_raw')
os.environ['nnUNet_preprocessed'] = os.path.join(WORKSPACE, 'nnUNet_preprocessed')

ENTORNO = {
    'python': sys.version.split()[0],
    'torch': torch.__version__,
    'nnunetv2': md.version('nnunetv2'),
    'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE',
}
print(TAG, ENTORNO)
assert torch.cuda.is_available(), 'No GPU in this session'


In [ ]:
# === Cell 2: gates on the inputs (nothing is trained if one fails) ===
import re

def read_stems(path):
    with open(path) as f:
        return sorted(l.strip() for l in f if l.strip())

TRAIN_10 = read_stems(os.path.join(BASE, 'label_fractions', 'frac_10', 'stems.txt'))
TRAIN_25 = read_stems(os.path.join(BASE, 'label_fractions', 'frac_25', 'stems.txt'))
VAL = sorted(f[:-4] for f in os.listdir(os.path.join(BASE, 'val', 'images')) if f.endswith('.png'))
TEST = sorted(f[:-4] for f in os.listdir(os.path.join(BASE, 'test', 'masks')) if f.endswith('.png'))
POOL_DIR = os.path.join(BASE, 'unlabeling_r10_max0', 'images')
POOL = sorted(f for f in os.listdir(POOL_DIR) if f.endswith('.png'))

DS501_RAW = os.path.join(os.environ['nnUNet_raw'], 'Dataset501_VFSS')
DS501_PRE = os.path.join(os.environ['nnUNet_preprocessed'], 'Dataset501_VFSS')

assert len(TRAIN_10) == 23, len(TRAIN_10)
assert len({s.split('_')[0] for s in TRAIN_10}) == 23, 'expected one frame per video'
assert set(TRAIN_10) <= set(TRAIN_25), 'frac_10 is not nested in frac_25'
assert len(VAL) == 44 and len(TEST) == 63, (len(VAL), len(TEST))
assert not set(TRAIN_10) & (set(VAL) | set(TEST)), 'train overlaps val/test'
train_videos = {s.split('_')[0] for s in TRAIN_10}
assert not train_videos & {s.split('_')[0] for s in VAL + TEST}, 'a train video is in val/test'
assert len(POOL) == 3937, len(POOL)
for s in TRAIN_10 + VAL:
    assert os.path.exists(os.path.join(DS501_RAW, 'imagesTr', f'{s}_0000.png')), s
PLAN_501 = json.load(open(os.path.join(DS501_PRE, 'nnUNetPlans.json')))['configurations']['2d']
assert PLAN_501['patch_size'] == [1024, 1024] and PLAN_501['batch_size'] == 3, PLAN_501['patch_size']
print('gates OK: 23 train (1 per video), 44 val, 63 test, pool 3937, plan 1024x1024 batch 3')


In [ ]:
# === Cell 3: helpers (resume-aware training, log gate, evaluation) ===
import csv, glob, shutil, time
import numpy as np
from PIL import Image

def fold_dir(results_dir, dataset_id):
    hits = glob.glob(os.path.join(results_dir, f'Dataset{dataset_id}_*', 'nnUNetTrainer__nnUNetPlans__2d', 'fold_0'))
    return hits[0] if hits else None

def write_split_501_frac10():
    """Dataset501 keeps one splits_final.json for every label fraction, so it is rewritten before each start."""
    path = os.path.join(DS501_PRE, 'splits_final.json')
    with open(path, 'w') as f:
        json.dump([{'train': TRAIN_10, 'val': VAL}], f, indent=2)
    back = json.load(open(path))[0]
    assert len(back['train']) == 23 and len(back['val']) == 44

def train(dataset_id, results_dir, before_start=None):
    """Train fold 0 for 1000 epochs. Skips a finished run and continues an interrupted one."""
    os.makedirs(results_dir, exist_ok=True)
    os.environ['nnUNet_results'] = results_dir
    fd = fold_dir(results_dir, dataset_id)
    if fd and os.path.exists(os.path.join(fd, 'checkpoint_final.pth')):
        print('already trained, skipping:', fd)
        return
    resume = bool(fd) and os.path.exists(os.path.join(fd, 'checkpoint_latest.pth'))
    if before_start:
        before_start()
    t0 = time.time()
    get_ipython().system(f'nnUNetv2_train {dataset_id} 2d 0' + (' --c' if resume else ''))
    print(f'training call returned after {(time.time() - t0) / 3600:.1f} h (resume={resume})')
    fd = fold_dir(results_dir, dataset_id)
    assert fd and os.path.exists(os.path.join(fd, 'checkpoint_final.pth')), 'training did not finish; rerun this cell to resume'

def assert_split_in_logs(results_dir, dataset_id, n_train):
    """Every training log of the run must report the expected split, including resumed segments."""
    logs = sorted(glob.glob(os.path.join(fold_dir(results_dir, dataset_id), 'training_log_*.txt')))
    found = []
    for lg in logs:
        m = re.search(r'This split has (\d+) training and (\d+) validation cases', open(lg, errors='ignore').read())
        if m:
            found.append((int(m.group(1)), int(m.group(2))))
    print('splits reported by the logs:', found)
    assert found and all(x == (n_train, 44) for x in found), f'expected ({n_train}, 44) in every log'

def predict(dataset_id, results_dir, in_dir, out_dir, probabilities=False):
    os.environ['nnUNet_results'] = results_dir
    if os.path.isdir(out_dir):
        shutil.rmtree(out_dir)
    os.makedirs(out_dir)
    extra = ' --save_probabilities' if probabilities else ''
    get_ipython().system(f'nnUNetv2_predict -i "{in_dir}" -o "{out_dir}" -d {dataset_id} -c 2d -f 0{extra}')

def evaluate(pred_dir, condition, extra):
    """Per-image Dice/IoU on the 63 test masks, same computation as cells A and G of notebook 02."""
    gt_dir = os.path.join(BASE, 'test', 'masks')
    preds = {re.sub(r'_\d{4}$', '', f[:-4]): f for f in os.listdir(pred_dir) if f.endswith('.png')}
    assert set(TEST) <= set(preds), f'missing predictions: {len(set(TEST) - set(preds))}'
    rows = []
    for stem in TEST:
        gt = np.array(Image.open(os.path.join(gt_dir, stem + '.png')).convert('L'))
        pr = np.array(Image.open(os.path.join(pred_dir, preds[stem])).convert('L'))
        if pr.shape != gt.shape:
            pr = np.array(Image.fromarray(pr).resize((gt.shape[1], gt.shape[0]), Image.NEAREST))
        g, p = (gt > 0).astype(np.float32), (pr > 0).astype(np.float32)
        inter, sp, sg = np.sum(p * g), np.sum(p), np.sum(g)
        dice = (2.0 * inter) / (sp + sg) if (sp + sg) > 0 else 1.0
        union = sp + sg - inter
        rows.append({'stem': stem, 'f1': dice, 'iou': inter / union if union > 0 else 1.0})
    with open(os.path.join(RESULTS_BASE, f'per_image_metrics_{condition}.csv'), 'w', newline='') as f:
        w = csv.DictWriter(f, fieldnames=['stem', 'f1', 'iou']); w.writeheader(); w.writerows(rows)
    metrics = {'condition': condition, 'run_id': RUN_ID,
               'mean_f1': float(np.mean([r['f1'] for r in rows])),
               'mean_iou': float(np.mean([r['iou'] for r in rows])),
               'num_test_images': len(rows), 'entorno': ENTORNO, **extra}
    with open(os.path.join(RESULTS_BASE, f'metrics_{condition}.json'), 'w') as f:
        json.dump(metrics, f, indent=2)
    print(f"{condition}: F1={metrics['mean_f1']:.4f}  IoU={metrics['mean_iou']:.4f}")
    return metrics

def done(condition):
    return os.path.exists(os.path.join(RESULTS_BASE, f'metrics_{condition}.json'))


In [ ]:
# === Cell 4: SUPERVISED nnU-Net, 23 labeled frames ===
SUP = f'sup_{TAG}'
SUP_RESULTS = os.path.join(WORKSPACE, f'nnUNet_results_{SUP}')

if done(SUP):
    print('already evaluated:', SUP)
else:
    train(501, SUP_RESULTS, before_start=write_split_501_frac10)
    assert_split_in_logs(SUP_RESULTS, 501, n_train=23)
    sup_pred = os.path.join(RESULTS_BASE, f'predictions_{SUP}')
    predict(501, SUP_RESULTS, os.path.join(DS501_RAW, 'imagesTs'), sup_pred)
    evaluate(sup_pred, SUP, {'num_train': 23, 'num_val': 44})


In [ ]:
# === Cell 5: pseudo-labels of THIS run's supervised model on the r10 pool ===
# Same rule as cell F of notebook 02: a frame is kept when the mean over pixels of max(p, 1-p) is >= 0.95.
CONF_THRESHOLD = 0.95
PSEUDO_DIR = os.path.join(RESULTS_BASE, f'pseudo_labels_{TAG}')
PSEUDO_STATS = os.path.join(RESULTS_BASE, f'pseudo_confidence_{TAG}.csv')

if os.path.exists(PSEUDO_STATS):
    print('pseudo-labels already generated:', PSEUDO_DIR)
else:
    assert_split_in_logs(SUP_RESULTS, 501, n_train=23)
    tmp_in, tmp_out = f'/content/_pool_in_{TAG}', f'/content/_pool_pred_{TAG}'
    if os.path.isdir(tmp_in):
        shutil.rmtree(tmp_in)
    os.makedirs(tmp_in)
    for fname in POOL:
        Image.open(os.path.join(POOL_DIR, fname)).convert('L').save(os.path.join(tmp_in, fname[:-4] + '_0000.png'))
    predict(501, SUP_RESULTS, tmp_in, tmp_out, probabilities=True)

    for sub in ('imagesTr', 'labelsTr'):
        d = os.path.join(PSEUDO_DIR, sub)
        if os.path.isdir(d):
            shutil.rmtree(d)
        os.makedirs(d)
    stats = []
    for npz in sorted(f for f in os.listdir(tmp_out) if f.endswith('.npz')):
        stem = npz[:-4]
        probs = np.load(os.path.join(tmp_out, npz))['probabilities']
        fg = np.asarray(probs[1] if probs.shape[0] >= 2 else probs[0]).squeeze()
        assert fg.ndim == 2, fg.shape
        conf = float(np.maximum(fg, 1.0 - fg).mean())
        mask = (fg >= 0.5).astype(np.uint8)
        kept = conf >= CONF_THRESHOLD
        stats.append({'stem': stem, 'mean_conf': conf, 'fg_pixels': int(mask.sum()), 'kept': int(kept)})
        if kept:
            ps = f'pseudo10r{RUN_ID}_{stem}'
            Image.fromarray(mask).save(os.path.join(PSEUDO_DIR, 'labelsTr', ps + '.png'))
            shutil.copy2(os.path.join(tmp_in, stem + '_0000.png'), os.path.join(PSEUDO_DIR, 'imagesTr', ps + '_0000.png'))
    assert len(stats) == len(POOL), (len(stats), len(POOL))
    with open(PSEUDO_STATS, 'w', newline='') as f:
        w = csv.DictWriter(f, fieldnames=list(stats[0])); w.writeheader(); w.writerows(stats)
    shutil.rmtree(tmp_in); shutil.rmtree(tmp_out)

st = list(csv.DictReader(open(PSEUDO_STATS)))
confs = np.array([float(r['mean_conf']) for r in st])
print(f"pool {len(st)} | kept {sum(int(r['kept']) for r in st)} | empty masks {sum(int(r['fg_pixels']) == 0 for r in st)}")
print('mean_conf  min %.4f  p5 %.4f  median %.4f  max %.4f' % (confs.min(), np.percentile(confs, 5), np.median(confs), confs.max()))


In [ ]:
# === Cell 6: SELF-TRAINING nnU-Net, 23 real + pseudo-labeled frames ===
SSL = f'ssl_{TAG}'
SSL_ID = 510 + RUN_ID
SSL_NAME = f'Dataset{SSL_ID}_VFSS_SSL10R{RUN_ID}'
SSL_RAW = os.path.join(os.environ['nnUNet_raw'], SSL_NAME)
SSL_PRE = os.path.join(os.environ['nnUNet_preprocessed'], SSL_NAME)
SSL_RESULTS = os.path.join(WORKSPACE, f'nnUNet_results_{SSL}')

if done(SSL):
    print('already evaluated:', SSL)
else:
    pseudo = sorted(f[:-9] for f in os.listdir(os.path.join(PSEUDO_DIR, 'imagesTr')) if f.endswith('_0000.png'))
    assert pseudo, 'no pseudo-labels were kept'
    split_train, n_cases = sorted(TRAIN_10 + pseudo), 23 + 44 + len(pseudo)

    raw_ready = os.path.isdir(os.path.join(SSL_RAW, 'imagesTr')) and len(os.listdir(os.path.join(SSL_RAW, 'imagesTr'))) == n_cases \
        and os.path.exists(os.path.join(SSL_RAW, 'dataset.json'))
    if not raw_ready:
        for sub in ('imagesTr', 'labelsTr', 'imagesTs'):
            d = os.path.join(SSL_RAW, sub)
            if os.path.isdir(d):
                shutil.rmtree(d)
            os.makedirs(d)
        for s in TRAIN_10 + VAL:
            shutil.copy2(os.path.join(DS501_RAW, 'imagesTr', f'{s}_0000.png'), os.path.join(SSL_RAW, 'imagesTr'))
            shutil.copy2(os.path.join(DS501_RAW, 'labelsTr', f'{s}.png'), os.path.join(SSL_RAW, 'labelsTr'))
        for s in pseudo:
            shutil.copy2(os.path.join(PSEUDO_DIR, 'imagesTr', f'{s}_0000.png'), os.path.join(SSL_RAW, 'imagesTr'))
            shutil.copy2(os.path.join(PSEUDO_DIR, 'labelsTr', f'{s}.png'), os.path.join(SSL_RAW, 'labelsTr'))
        for fname in os.listdir(os.path.join(DS501_RAW, 'imagesTs')):
            shutil.copy2(os.path.join(DS501_RAW, 'imagesTs', fname), os.path.join(SSL_RAW, 'imagesTs'))
        with open(os.path.join(SSL_RAW, 'dataset.json'), 'w') as f:
            json.dump({'channel_names': {'0': 'Xray'}, 'labels': {'background': 0, 'vertebra': 1},
                       'numTraining': n_cases, 'file_ending': '.png'}, f, indent=2)
    print(f'{SSL_NAME}: {n_cases} cases = 23 real + 44 val + {len(pseudo)} pseudo')

    pre_dir = os.path.join(SSL_PRE, 'nnUNetPlans_2d')
    pre_ready = os.path.isdir(pre_dir) and len([f for f in os.listdir(pre_dir) if f.endswith('.pkl')]) == n_cases
    if not pre_ready:
        get_ipython().system(f'nnUNetv2_plan_and_preprocess -d {SSL_ID} --verify_dataset_integrity -c 2d')
    plan = json.load(open(os.path.join(SSL_PRE, 'nnUNetPlans.json')))['configurations']['2d']
    for k in ('patch_size', 'batch_size', 'normalization_schemes'):
        assert plan[k] == PLAN_501[k], f'plan differs from Dataset501 in {k}: {plan[k]} vs {PLAN_501[k]}'
    assert plan['architecture']['arch_kwargs'] == PLAN_501['architecture']['arch_kwargs'], 'architecture differs from Dataset501'

    def write_split_ssl():
        with open(os.path.join(SSL_PRE, 'splits_final.json'), 'w') as f:
            json.dump([{'train': split_train, 'val': VAL}], f, indent=2)

    train(SSL_ID, SSL_RESULTS, before_start=write_split_ssl)
    assert_split_in_logs(SSL_RESULTS, SSL_ID, n_train=len(split_train))
    ssl_pred = os.path.join(RESULTS_BASE, f'predictions_{SSL}')
    predict(SSL_ID, SSL_RESULTS, os.path.join(SSL_RAW, 'imagesTs'), ssl_pred)
    evaluate(ssl_pred, SSL, {'real_train': 23, 'pseudo_kept': len(pseudo), 'pool': len(POOL), 'conf_threshold': CONF_THRESHOLD})


In [ ]:
# === Cell 7: summary of every finished f10 run (raw numbers, all repetitions on Drive) ===
import pandas as pd

rows = []
for k in range(3):
    a = os.path.join(RESULTS_BASE, f'per_image_metrics_sup_f10_run{k}.csv')
    b = os.path.join(RESULTS_BASE, f'per_image_metrics_ssl_f10_run{k}.csv')
    sup = pd.read_csv(a).set_index('stem').f1 if os.path.exists(a) else None
    ssl = pd.read_csv(b).set_index('stem').f1 if os.path.exists(b) else None
    rows.append({'run': k,
                 'sup_f1': None if sup is None else round(sup.mean(), 4),
                 'ssl_f1': None if ssl is None else round(ssl.mean(), 4),
                 'delta': None if sup is None or ssl is None else round((ssl - sup).mean(), 4),
                 'improved_of_63': None if sup is None or ssl is None else int(((ssl - sup) > 0).sum())})
tab = pd.DataFrame(rows)
print(tab.to_string(index=False))
full = tab.dropna()
if len(full) >= 2:
    print(f"\nsup  {full.sup_f1.mean():.4f} +/- {full.sup_f1.std(ddof=1):.4f}   ssl  {full.ssl_f1.mean():.4f} +/- {full.ssl_f1.std(ddof=1):.4f}   (n={len(full)} runs, sample std)")
    print('The spread of sup_f1 across runs is the run-to-run noise that any delta has to exceed.')
print('Reference, single runs: nnU-Net 25% sup .8962 / self-training .9013;  100% sup .9075')


In [ ]:
# === Cell 8: optional, leave Dataset501 on the 100% split ===
# Dataset501/splits_final.json is shared by every label fraction. Set this to True ONLY when no other
# f10 notebook is still training or may need to resume, otherwise that run would restart on 218 frames
# (the log gate of cell 4 would catch it, but the hours would be lost).
RESTORE_FULL_SPLIT = False

if RESTORE_FULL_SPLIT:
    full = sorted(f[:-4] for f in os.listdir(os.path.join(BASE, 'train', 'images')) if f.endswith('.png'))
    assert len(full) == 218, len(full)
    with open(os.path.join(DS501_PRE, 'splits_final.json'), 'w') as f:
        json.dump([{'train': full, 'val': VAL}], f, indent=2)
    print('Dataset501 split restored: 218 train + 44 val')
else:
    n = len(json.load(open(os.path.join(DS501_PRE, 'splits_final.json')))[0]['train'])
    print(f'Dataset501 split left as is: {n} train cases')
